# Hybrid Model Pipeline
## Graph + Text Feature Engineering for Fake Review Detection

**Pipeline Steps:**
1. Load preprocessed data (nodes, edges, features, embeddings)
2. Build graph structure for GNN
3. Combine text embeddings with structured features
4. Prepare training data (graph + text + weak labels)
5. Create data loaders for model training

## 1. Setup & Imports

In [1]:
import sys
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd
import json
from collections import defaultdict
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ Libraries imported")

✓ Libraries imported


In [2]:
# === SETUP PATHS ===
PROJECT_DIR = Path(".")
# .resolve() giúp in ra đường dẫn tuyệt đối thực tế để bạn quan sát
print(f"Data directory: {PROJECT_DIR.resolve()}")
print(f"\nFiles available:")

GOLD_DIR = PROJECT_DIR / "data" / "gold"
CATEGORY = "All_Beauty"
DATA_DIR = GOLD_DIR / CATEGORY

print(f"Project Directory: {PROJECT_DIR}")
print(f"Data Directory: {DATA_DIR}")

# Create output directory for hybrid model data
HYBRID_DIR = DATA_DIR / "hybrid_model"
HYBRID_DIR.mkdir(parents=True, exist_ok=True)
print(f"Hybrid Model Directory: {HYBRID_DIR}")

Data directory: E:\MXH

Files available:
Project Directory: .
Data Directory: data\gold\All_Beauty
Hybrid Model Directory: data\gold\All_Beauty\hybrid_model


## 2. Load Preprocessed Data

In [3]:
# === LOAD NODES ===
logger.info("Loading node tables...")
nodes_user = pd.read_parquet(DATA_DIR / "nodes_user.parquet")
nodes_review = pd.read_parquet(DATA_DIR / "nodes_review.parquet")
nodes_item = pd.read_parquet(DATA_DIR / "nodes_item.parquet")

print(f"✓ Users: {len(nodes_user)}")
print(f"✓ Reviews: {len(nodes_review)}")
print(f"✓ Items: {len(nodes_item)}")

2026-06-09 20:31:16,489 - INFO - Loading node tables...


✓ Users: 631986
✓ Reviews: 693929
✓ Items: 112565


In [4]:
# === LOAD EDGES ===
logger.info("Loading edge tables...")
edges_user_review = pd.read_parquet(DATA_DIR / "edges_user_review.parquet")
edges_review_item = pd.read_parquet(DATA_DIR / "edges_review_item.parquet")
edges_user_item = pd.read_parquet(DATA_DIR / "edges_user_item.parquet")
edges_review_review = pd.read_parquet(DATA_DIR / "edges_review_review.parquet")

print(f"✓ User-Review edges: {len(edges_user_review)}")
print(f"✓ Review-Item edges: {len(edges_review_item)}")
print(f"✓ User-Item edges: {len(edges_user_item)}")
print(f"✓ Review-Review edges: {len(edges_review_review)}")

2026-06-09 20:31:17,715 - INFO - Loading edge tables...


✓ User-Review edges: 693929
✓ Review-Item edges: 693929
✓ User-Item edges: 693929
✓ Review-Review edges: 296494


In [5]:
# === CELL 4: LOAD FEATURES & LABELS RAW ===
logger.info("Loading features and labels...")
review_features = pd.read_parquet(DATA_DIR / "review_features.parquet")
review_splits = pd.read_parquet(DATA_DIR / "review_splits.parquet")

print(f"✓ Review features: {review_features.shape}")
print(f"✓ Review splits: {review_splits.shape}")
print(f"\nSplit distribution:")
print(review_splits["split"].value_counts())
print(f"\nWeak label distribution:")
print(review_splits["weak_label"].value_counts())

2026-06-09 20:31:19,262 - INFO - Loading features and labels...


✓ Review features: (693929, 39)
✓ Review splits: (693929, 4)

Split distribution:
train    485750
test     104090
val      104089
Name: split, dtype: int64

Weak label distribution:
1    523069
0    170860
Name: weak_label, dtype: int64


In [6]:
# === LOAD TEXT EMBEDDINGS ===
logger.info("Loading text embeddings...")
embedding_path = DATA_DIR / f"review_text_embeddings_{CATEGORY}.npy"

if embedding_path.exists():
    review_embeddings = np.load(embedding_path)
    print(f"✓ Embeddings shape: {review_embeddings.shape}")
    print(f"  - Reviews: {review_embeddings.shape[0]}")
    print(f"  - Embedding dimension: {review_embeddings.shape[1]}")
else:
    print("⚠ No embeddings found - will create dummy embeddings")
    review_embeddings = None

2026-06-09 20:31:20,056 - INFO - Loading text embeddings...


✓ Embeddings shape: (693929, 384)
  - Reviews: 693929
  - Embedding dimension: 384


## 3. Build Graph Structure

In [ ]:
class GraphBuilder:
    """Build heterogeneous graph for GNN."""
    
    def __init__(self, nodes_user, nodes_review, nodes_item, edges):
        self.nodes_user = nodes_user
        self.nodes_review = nodes_review
        self.nodes_item = nodes_item
        
        # Create ID mappings
        self.user_id_map = {uid: i for i, uid in enumerate(nodes_user['user_id'])}
        self.review_id_map = {rid: i for i, rid in enumerate(nodes_review['review_id'])}
        self.item_id_map = {iid: i for i, iid in enumerate(nodes_item['item_id'])}
        
        self.edges = edges
        logger.info(f"Graph built with {len(self.user_id_map)} users, {len(self.review_id_map)} reviews, {len(self.item_id_map)} items")
    
    def get_adjacency_matrix(self, edge_type: str) -> Tuple[List[Tuple], str]:
        """
        Get edge list and node types for specific edge type.
        """
        edge_data = self.edges[edge_type]
        edges_list = []
        
        if edge_type == 'edges_user_review':
            for _, row in edge_data.iterrows():
                if row['src'] in self.user_id_map and row['dst'] in self.review_id_map:
                    edges_list.append((self.user_id_map[row['src']], self.review_id_map[row['dst']]))
            node_types = ("user", "review")
            
        elif edge_type == 'edges_review_item':
            for _, row in edge_data.iterrows():
                if row['src'] in self.review_id_map and row['dst'] in self.item_id_map:
                    edges_list.append((self.review_id_map[row['src']], self.item_id_map[row['dst']]))
            node_types = ("review", "item")
            
        elif edge_type == 'edges_user_item':
            for _, row in edge_data.iterrows():
                if row['src'] in self.user_id_map and row['dst'] in self.item_id_map:
                    edges_list.append((self.user_id_map[row['src']], self.item_id_map[row['dst']]))
            node_types = ("user", "item")
            
        elif edge_type == 'edges_review_review':
            for _, row in edge_data.iterrows():
                if row['src'] in self.review_id_map and row['dst'] in self.review_id_map:
                    edges_list.append((self.review_id_map[row['src']], self.review_id_map[row['dst']]))
            node_types = ("review", "review")
        
        return edges_list, node_types
    
    def get_graph_dict(self) -> Dict:
        """
        Get complete graph as dictionary with all edge types.
        """
        graph = {}
        for edge_type_key in self.edges.keys():
            edges_list, node_types = self.get_adjacency_matrix(edge_type_key)
            if len(edges_list) > 0:
                graph[edge_type_key] = {
                    'edges': edges_list,
                    'node_types': node_types,
                    'num_edges': len(edges_list)
                }
        return graph

# Build graph
edges_dict = {
    'edges_user_review': edges_user_review,
    'edges_review_item': edges_review_item,
    'edges_user_item': edges_user_item,
    'edges_review_review': edges_review_review
}

# Khởi tạo GraphBuilder với danh sách nút đã đồng bộ hóa
graph_builder = GraphBuilder(nodes_user, nodes_review, nodes_item, edges_dict)
graph_dict = graph_builder.get_graph_dict()

logger.info(f"Graph structure built with {len(graph_dict)} edge types")
for edge_type, data in graph_dict.items():
    print(f"  {edge_type}: {data['num_edges']} edges ({data['node_types']})")

NameError: name 'reviews_clean' is not defined

## 4. Prepare Hybrid Features (Graph + Text)

In [ ]:
# === CELL 7: HYBRID FEATURE BUILDER WITH ZERO-PADDING OPTIMIZATION ===
class HybridFeatureBuilder:
    """Combine graph structural features with text embeddings using zero-padding for alignment."""
    
    def __init__(self, review_features, review_embeddings, nodes_review, review_splits):
        self.review_features = review_features
        self.review_embeddings = review_embeddings
        self.nodes_review = nodes_review
        self.review_splits = review_splits
        self.total_nodes = len(nodes_review)  # Đảm bảo giữ đúng 693,929 nút
        
    def get_structural_features(self) -> np.ndarray:
        """Extract structured features from review_features."""
        exclude_cols = ['review_id', 'weak_label', 'suspicion_score']
        feature_cols = [col for col in self.review_features.columns if col not in exclude_cols]
        
        structural_features = self.review_features[feature_cols].values.astype(np.float32)
        structural_features = np.nan_to_num(structural_features, nan=0.0, posinf=0.0, neginf=0.0)
        
        logger.info(f"Structural features shape: {structural_features.shape}")
        return structural_features, feature_cols
    
    def get_text_embeddings(self) -> np.ndarray:
        """Get text embeddings and pad missing rows with zeros."""
        # Khởi tạo ma trận zero toàn phần kích thước (693929, 384)
        text_emb = np.zeros((self.total_nodes, 384), dtype=np.float32)
        
        if self.review_embeddings is not None:
            # Điền các vector nhúng văn bản đang có vào phần đầu ma trận
            available_len = min(len(self.review_embeddings), self.total_nodes)
            text_emb[:available_len] = self.review_embeddings[:available_len]
            
            # Chuẩn hóa L2-norm cho các hàng chứa dữ liệu thực tế
            norms = np.linalg.norm(text_emb, axis=1, keepdims=True)
            norms[norms == 0] = 1
            text_emb = text_emb / norms
        else:
            logger.warning("No embeddings available - creating dummy embeddings")
            text_emb = np.random.randn(self.total_nodes, 384).astype(np.float32)
            
        logger.info(f"Text embeddings shape (aligned with padding): {text_emb.shape}")
        return text_emb
    
    def combine_features(self, structural_features, text_embeddings) -> np.ndarray:
        """Combine structural features, advanced anomaly scores and text embeddings."""
        # TÍCH HỢP NÂNG CAO: Tự động nạp và bù zero cho điểm số dị thường ở Giai đoạn 2
        try:
            ANOMALY_DIR = DATA_DIR / "anomaly_detection"
            if (ANOMALY_DIR / "hybrid_anomaly_scores.npy").exists():
                hybrid_scores = np.load(ANOMALY_DIR / "hybrid_anomaly_scores.npy")
                graph_suspicion = np.load(ANOMALY_DIR / "graph_suspicion.npy")
                semantic_suspicion = np.load(ANOMALY_DIR / "semantic_suspicion.npy")
                
                # Tạo ma trận đệm zero bảo vệ hệ thống
                advanced_scores = np.zeros((self.total_nodes, 3), dtype=np.float32)
                l = min(len(hybrid_scores), self.total_nodes)
                advanced_scores[:l, 0] = hybrid_scores[:l]
                advanced_scores[:l, 1] = graph_suspicion[:l]
                advanced_scores[:l, 2] = semantic_suspicion[:l]
                
                # Nối điểm số dị thường vào nhóm đặc trưng cấu trúc đồ thị
                structural_features = np.concatenate([structural_features, advanced_scores], axis=1)
                logger.info("🔥 Tích hợp thành công Tín hiệu dị thường nâng cao Giai đoạn 2 vào ma trận đặc trưng!")
        except Exception as e:
            logger.warning(f"Không thể nạp tín hiệu dị thường bổ sung: {e}")

        combined = np.concatenate([structural_features, text_embeddings], axis=1)
        logger.info(f"Combined features shape final: {combined.shape}")
        return combined
    
    def get_labels_and_splits(self):
        """Get weak labels and train/val/test splits."""
        labels = self.review_splits['weak_label'].values.astype(np.int32)
        split_map = {'train': 0, 'val': 1, 'test': 2}
        splits = self.review_splits['split'].map(split_map).values.astype(np.int32)
        
        return labels, splits

# Tiến hành khởi tạo và xử lý trích xuất đặc trưng lai lai
feature_builder = HybridFeatureBuilder(review_features, review_embeddings, nodes_review, review_splits)

structural_features, feature_cols = feature_builder.get_structural_features()
text_embeddings = feature_builder.get_text_embeddings()
combined_features = feature_builder.combine_features(structural_features, text_embeddings)
labels, splits = feature_builder.get_labels_and_splits()

print(f"\n✓ Hybrid features ready: {combined_features.shape}")

2026-06-09 20:28:53,606 - INFO - Loading features and labels...


✓ Review features: (693929, 39)
✓ Review splits: (693929, 4)

Split distribution:
train    485750
test     104090
val      104089
Name: split, dtype: int64

Weak label distribution:
1    523069
0    170860
Name: weak_label, dtype: int64


## 5. Create Training Data Package

In [ ]:
class TrainingDataPackage:
    """Package all data needed for model training."""
    
    def __init__(self, graph_dict, combined_features, labels, splits, nodes_review, review_id_map):
        self.graph_dict = graph_dict
        self.combined_features = combined_features
        self.labels = labels
        self.splits = splits
        self.nodes_review = nodes_review
        self.review_id_map = review_id_map
    
    def get_split_indices(self):
        """
        Get indices for train/val/test splits.
        """
        train_idx = np.where(self.splits == 0)[0]
        val_idx = np.where(self.splits == 1)[0]
        test_idx = np.where(self.splits == 2)[0]
        
        return {
            'train': train_idx,
            'val': val_idx,
            'test': test_idx
        }
    
    def summary(self):
        """
        Print training data summary.
        """
        split_indices = self.get_split_indices()
        
        print("\n" + "="*60)
        print("TRAINING DATA PACKAGE SUMMARY")
        print("="*60)
        print(f"\nFeatures:")
        print(f"  Total reviews: {len(self.nodes_review)}")
        print(f"  Combined feature dimension: {self.combined_features.shape[1]}")
        print(f"\nGraph:")
        for edge_type, data in self.graph_dict.items():
            print(f"  {edge_type}: {data['num_edges']} edges")
        print(f"\nLabels (Weak Label Distribution):")
        unique, counts = np.unique(self.labels, return_counts=True)
        for label, count in zip(unique, counts):
            pct = count / len(self.labels) * 100
            print(f"  Label {label}: {count:,} ({pct:.1f}%)")
        print(f"\nData Split:")
        print(f"  Train: {len(split_indices['train']):,} ({len(split_indices['train'])/len(self.labels)*100:.1f}%)")
        print(f"  Val:   {len(split_indices['val']):,} ({len(split_indices['val'])/len(self.labels)*100:.1f}%)")
        print(f"  Test:  {len(split_indices['test']):,} ({len(split_indices['test'])/len(self.labels)*100:.1f}%)")
        print("="*60)

# Create training data package
training_pkg = TrainingDataPackage(
    graph_dict,
    combined_features,
    labels,
    splits,
    nodes_review,
    graph_builder.review_id_map
)

training_pkg.summary()


TRAINING DATA PACKAGE SUMMARY

Features:
  Total reviews: 693929
  Combined feature dimension: 420

Graph:
  edges_user_item: 693929 edges

Labels (Weak Label Distribution):

Data Split:


ZeroDivisionError: division by zero

## 6. Save Training Data

In [ ]:
# === SAVE FEATURES ===
logger.info(f"Saving features to {HYBRID_DIR}...")

np.save(HYBRID_DIR / "combined_features.npy", combined_features)
np.save(HYBRID_DIR / "labels.npy", labels)
np.save(HYBRID_DIR / "splits.npy", splits)

logger.info("✓ Features saved")

2026-06-09 16:00:58,854 - INFO - Saving features to data\gold\All_Beauty\hybrid_model...
2026-06-09 16:01:00,139 - INFO - ✓ Features saved


In [ ]:
# === SAVE GRAPH ===
logger.info("Saving graph structure...")

graph_json = {}
for edge_type, data in graph_dict.items():
    graph_json[edge_type] = {
        'edges': [[int(u), int(v)] for u, v in data['edges']],
        'node_types': data['node_types'],
        'num_edges': data['num_edges']
    }

with open(HYBRID_DIR / "graph_structure.json", "w") as f:
    json.dump(graph_json, f, indent=2)

logger.info("✓ Graph saved")

2026-06-09 16:01:00,166 - INFO - Saving graph structure...
2026-06-09 16:01:13,101 - INFO - ✓ Graph saved


In [ ]:
# === SAVE METADATA ===
logger.info("Saving metadata...")

metadata = {
    'num_reviews': len(nodes_review),
    'num_users': len(nodes_user),
    'num_items': len(nodes_item),
    'feature_dimension': combined_features.shape[1],
    'structural_features': structural_features.shape[1],
    'text_embedding_dim': text_embeddings.shape[1],
    'num_edge_types': len(graph_dict),
    'total_edges': sum(data['num_edges'] for data in graph_dict.values()),
    'label_distribution': {
        str(int(k)): int(v) for k, v in zip(*np.unique(labels, return_counts=True))
    },
    'split_distribution': {
        'train': int(np.sum(splits == 0)),
        'val': int(np.sum(splits == 1)),
        'test': int(np.sum(splits == 2))
    }
}

with open(HYBRID_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

logger.info("✓ Metadata saved")

2026-06-09 16:01:13,130 - INFO - Saving metadata...
2026-06-09 16:01:13,143 - INFO - ✓ Metadata saved


In [ ]:
# === SAVE NODE MAPPINGS ===
logger.info("Saving ID mappings...")

mappings = {
    'review_id_map': {k: v for k, v in graph_builder.review_id_map.items()},
    'user_id_map': {k: v for k, v in graph_builder.user_id_map.items()},
    'item_id_map': {k: v for k, v in graph_builder.item_id_map.items()}
}

with open(HYBRID_DIR / "id_mappings.json", "w") as f:
    json.dump(mappings, f)

logger.info("✓ ID mappings saved")

2026-06-09 16:01:13,158 - INFO - Saving ID mappings...
2026-06-09 16:01:16,642 - INFO - ✓ ID mappings saved


## 7. Verification & Summary

In [ ]:
# === VERIFY SAVED FILES ===
print("\nSaved files:")
for file in sorted(HYBRID_DIR.iterdir()):
    if file.is_file():
        if file.suffix == '.npy':
            size = file.stat().st_size / (1024 * 1024)
            arr = np.load(file)
            print(f"  ✓ {file.name:<35} {arr.shape} ({size:.2f} MB)")
        elif file.suffix == '.json':
            size = file.stat().st_size / 1024
            print(f"  ✓ {file.name:<35} ({size:.2f} KB)")


Saved files:
  ✓ combined_features.npy               (693929, 420) (1111.79 MB)
  ✓ graph_structure.json                (118624.69 KB)
  ✓ id_mappings.json                    (56597.24 KB)
  ✓ labels.npy                          (693929,) (2.65 MB)
  ✓ metadata.json                       (0.37 KB)
  ✓ splits.npy                          (693929,) (2.65 MB)


In [ ]:
# === FINAL SUMMARY ===
print("\n" + "="*70)
print("HYBRID MODEL PIPELINE COMPLETE")
print("="*70)
print(f"\nOutput Directory: {HYBRID_DIR}")
print(f"\nDataset Statistics:")
print(f"  Total Samples: {len(labels):,}")
print(f"  Feature Dimension: {combined_features.shape[1]}")
print(f"    - Structural Features: {structural_features.shape[1]}")
print(f"    - Text Embeddings: {text_embeddings.shape[1]}")
print(f"\nGraph Statistics:")
for edge_type, data in graph_dict.items():
    print(f"  {edge_type}: {data['num_edges']:,}")
print(f"\nClass Distribution:")
unique_labels, label_counts = np.unique(labels, return_counts=True)
for label, count in zip(unique_labels, label_counts):
    pct = count / len(labels) * 100
    print(f"  Class {label}: {count:,} ({pct:.1f}%)")
print(f"\nData Splits:")
print(f"  Train: {np.sum(splits == 0):,} samples")
print(f"  Val:   {np.sum(splits == 1):,} samples")
print(f"  Test:  {np.sum(splits == 2):,} samples")
print("\nReady for Model Training! 🚀")
print("="*70)


HYBRID MODEL PIPELINE COMPLETE

Output Directory: data\gold\All_Beauty\hybrid_model

Dataset Statistics:
  Total Samples: 693,929
  Feature Dimension: 420
    - Structural Features: 36
    - Text Embeddings: 384

Graph Statistics:
  edges_user_review: 693,929
  edges_review_item: 693,929
  edges_user_item: 693,929
  edges_review_review: 296,494

Class Distribution:
  Class 0: 170,860 (24.6%)
  Class 1: 523,069 (75.4%)

Data Splits:
  Train: 485,750 samples
  Val:   104,089 samples
  Test:  104,090 samples

Ready for Model Training! 🚀


## 8. Data Loading Example

In [ ]:
# === EXAMPLE: HOW TO LOAD DATA FOR TRAINING ===
print("Example code to load data for training:\n")
print("""
# Load hybrid model data
from pathlib import Path
import numpy as np
import json

HYBRID_DIR = Path("/Users/mac/Downloads/MXH FINAL/data/gold/All_Beauty/hybrid_model")

# Load features
features = np.load(HYBRID_DIR / "combined_features.npy")  # (N, D)
labels = np.load(HYBRID_DIR / "labels.npy")  # (N,)
splits = np.load(HYBRID_DIR / "splits.npy")  # (N,) - 0:train, 1:val, 2:test

# Load graph
with open(HYBRID_DIR / "graph_structure.json") as f:
    graph = json.load(f)

# Load metadata
with open(HYBRID_DIR / "metadata.json") as f:
    metadata = json.load(f)

# Get train/val/test indices
train_idx = np.where(splits == 0)[0]
val_idx = np.where(splits == 1)[0]
test_idx = np.where(splits == 2)[0]

# Create data loaders (PyTorch example)
from torch.utils.data import TensorDataset, DataLoader
import torch

X_train = torch.FloatTensor(features[train_idx])
y_train = torch.LongTensor(labels[train_idx])
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
""")

Example code to load data for training:


# Load hybrid model data
from pathlib import Path
import numpy as np
import json

HYBRID_DIR = Path("/Users/mac/Downloads/MXH FINAL/data/gold/All_Beauty/hybrid_model")

# Load features
features = np.load(HYBRID_DIR / "combined_features.npy")  # (N, D)
labels = np.load(HYBRID_DIR / "labels.npy")  # (N,)
splits = np.load(HYBRID_DIR / "splits.npy")  # (N,) - 0:train, 1:val, 2:test

# Load graph
with open(HYBRID_DIR / "graph_structure.json") as f:
    graph = json.load(f)

# Load metadata
with open(HYBRID_DIR / "metadata.json") as f:
    metadata = json.load(f)

# Get train/val/test indices
train_idx = np.where(splits == 0)[0]
val_idx = np.where(splits == 1)[0]
test_idx = np.where(splits == 2)[0]

# Create data loaders (PyTorch example)
from torch.utils.data import TensorDataset, DataLoader
import torch

X_train = torch.FloatTensor(features[train_idx])
y_train = torch.LongTensor(labels[train_idx])
train_dataset = TensorDataset(X_train, y_train)
tra